# ネプリライシン阻害剤複合体 (1R1H/1R1I/1R1J) での chem.protein.split() デモ

同じ標的タンパク質(ネプリライシン, neutral endopeptidase/NEP)に異なる低分子阻害剤が結合した
RCSBの3構造 `1R1H`/`1R1I`/`1R1J` を題材に、新しく実装した `chem.protein.split()` の動作を確認する。
3構造とも単一チェーンA、糖鎖修飾 `NAG` ×3・亜鉛イオン `ZN` ×1・低分子阻害剤(それぞれ
`BIR`/`TI1`/`OIR`)という同じ構成のヘテロ原子グループを持ち、「リガンドを含まないPDB」と
「水以外の各HETATM残基インスタンス全てのSDF」への分割を、複数構造にわたって横断的に
確認するのに適している。


In [ ]:
from chem import rcsb

entry_ids = ["1R1H", "1R1I", "1R1J"]
rcsb.download_structures(entry_ids, outdir="neprilysin_data", filetype="pdb")


## chem.protein.split() で構造を分割する

`chem.protein.split()` は構造ファイルを以下の2つに分割する:

1. **リガンドフリーの蛋白質PDB** -- 結晶水は残したまま、それ以外のHETATM残基
   (低分子リガンドはもちろん、亜鉛イオンや糖鎖修飾も)を全て取り除いたもの。
   デフォルト(`all_chains=False`)ではチェーンごとに、ファイル名にチェーンIDを
   含めて分割される(`all_chains=True` にすると全チェーンをまとめた1ファイルになる)
2. **リガンドSDF** -- 水を除く各HETATM残基インスタンス**全て**について1ファイル、
   例外なく必ず書き出す。3D座標は元の構造そのまま。PDB Chemical Component
   Dictionaryのテンプレートと照合して結合次数・芳香族性を復元できたものは
   `bond_orders_restored=True`(`chem.ligand.load_ligand` と同じロジック)、
   共有結合した糖鎖修飾やペプチド様リガンドなどでテンプレートと原子数が
   一致せず復元できなかったものは、警告付きで `bond_orders_restored=False`
   として原子座標間の距離から推定した単結合のみ・芳香族性なしの生の結合情報を書き出す
   (どちらの場合もインスタンスがスキップされることはない)


In [ ]:
import os

from chem import protein

split_results = {}
for entry_id in entry_ids:
    split_results[entry_id] = protein.split(
        os.path.join("neprilysin_data", f"{entry_id}.pdb"),
        outdir="neprilysin_split",
    )
split_results["1R1H"]


### 分割結果の一覧

各構造の `split()` が返した `"ligands"` を1つの表にまとめる。3構造とも `NAG`
(N-結合型糖鎖、3残基)は `bond_orders_restored=False`(RDKitのテンプレートマッチングの
既知の制限 -- 遊離型テンプレートに対し、糖鎖結合部位の原子が構造上欠けているため)、
`ZN` と主要阻害剤(`BIR`/`TI1`/`OIR`)は `bond_orders_restored=True` になることを確認する。


In [ ]:
import pandas as pd

rows = [
    {"entry_id": entry_id, **lig}
    for entry_id in entry_ids
    for lig in split_results[entry_id]["ligands"]
]
ligands_df = pd.DataFrame(rows)[
    ["entry_id", "code", "chain", "resnum", "icode", "bond_orders_restored", "path"]
]
ligands_df


### 水以外のHETATM残基が1つも欠けていないことを確認

`chem.ligand.list_ligand_instances` (`exclude=chem.protein.WATER`) で構造ファイル自身から
数えた水以外のHETATM残基インスタンス数と、`split()` が実際に書き出したSDF数が
一致する(＝結合次数を復元できないインスタンスも含め、1つも欠けずSDF化されている)ことを確認する。


In [ ]:
from chem.ligand import list_ligand_instances
from chem.protein import WATER

for entry_id in entry_ids:
    all_instances = list_ligand_instances(os.path.join("neprilysin_data", f"{entry_id}.pdb"), exclude=WATER)
    n_written = len(split_results[entry_id]["ligands"])
    print(entry_id, f"{n_written}/{len(all_instances)} non-water HETATM instances written to SDF")
    assert n_written == len(all_instances)


### リガンドフリー蛋白質PDBの中身を確認

デフォルト(`all_chains=False`)では `"protein"` はチェーンごとの `{チェーンID: パス}` 辞書になる
(3構造とも単一チェーンAのみなので `{"A": ...}`)。その各ファイルに改めて `list_ligand_instances`
を実行し、水以外のHETATM残基が本当に1つも残っていないことを確認する。


In [ ]:
for entry_id in entry_ids:
    for chain_id, protein_path in split_results[entry_id]["protein"].items():
        remaining = list_ligand_instances(protein_path, exclude=WATER)
        print(entry_id, chain_id, "remaining non-water HETATM residues:", remaining)


### 3種類の阻害剤SDFを比較する(分子量・QED・芳香族性)

主要阻害剤(`BIR`/`TI1`/`OIR`)のSDFを読み込み直し、芳香環が `GetIsAromatic()` で
芳香族として認識される(=結合次数が正しく復元されている)ことと、`chem.ligand` の
物性計算関数(`molecular_weight`/`qed`)をまとめて確認する。


In [ ]:
from rdkit import Chem

from chem import ligand

main_ligand_code = {"1R1H": "BIR", "1R1I": "TI1", "1R1J": "OIR"}

rows = []
for entry_id, code in main_ligand_code.items():
    lig_entry = next(l for l in split_results[entry_id]["ligands"] if l["code"] == code)
    mol = next(Chem.SDMolSupplier(lig_entry["path"]))
    rows.append(
        {
            "entry_id": entry_id,
            "code": code,
            "n_atoms": mol.GetNumAtoms(),
            "n_aromatic_atoms": sum(atom.GetIsAromatic() for atom in mol.GetAtoms()),
            "molecular_weight": round(ligand.molecular_weight(mol), 1),
            "qed": round(ligand.qed(mol), 3),
            "smiles": Chem.MolToSmiles(mol),
        }
    )
pd.DataFrame(rows)


### 可視化: リガンドフリー蛋白質 + 分割後のSDFを重ねて表示

`split()` はリガンドの3D座標を元の構造からそのままコピーするだけなので、分割後の蛋白質PDBと
リガンドSDFを同じ py3Dmol ビューに重ねれば、座標系がずれずに元の複合体そのままの配置で
表示できるはずである。ここでは `1R1H` のリガンドフリー蛋白質(チェーンA、cartoon)と
`BIR` 阻害剤のSDF(stick)を重ねてそれを確認する。


In [ ]:
import py3Dmol

protein_path = split_results["1R1H"]["protein"]["A"]
bir_path = next(l["path"] for l in split_results["1R1H"]["ligands"] if l["code"] == "BIR")

with open(protein_path) as f:
    protein_pdb_text = f.read()
with open(bir_path) as f:
    bir_sdf_text = f.read()

view = py3Dmol.view(width=600, height=450)
view.addModel(protein_pdb_text, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey"}})
view.addModel(bir_sdf_text, "sdf")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo({"model": 1})
view.show()
